In [3]:
#Enable L4 GPU in colab
# Check environment
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

NVIDIA L4, 23034 MiB


In NB3/NB4 we needed:
- `protobuf==3.20.3` — for TFRecord parsing
- `waymo-open-dataset-tf-2-11-0` — for Waymo frame extraction
- `crcmod` — for GCS file integrity checks

In NB5 we do NOT extract TFRecords — we restore the
already-prepared 150-segment dataset directly from GCS.

Only `ultralytics` is needed for YOLO26x training.

In [4]:
#Restoring prepared images from GCS and training.

#for YOLO26x training
!pip install ultralytics -q
print("Ultralytics installed ✅")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.4 MB/s eta 0:00:00
Ultralytics installed ✅


Import Path makes it easy to:

Create directories with mkdir()

Join paths with / operator like LOCAL_DATA_DIR / "images"

Pass path to YAML file as str(yaml_path)

Without Path you'd have to use raw strings everywhere like "/content/waymo_yolo/images/train" — more error prone.

,,,,,,

YOLO is the main class from the Ultralytics library that:

Loads the model: YOLO("yolo26x.pt")

Adds callbacks: model.add_callback(...)

Runs training: model.train(...)

Runs validation: model.val(...)

In [5]:
import os # for file existence checks only
from pathlib import Path
from ultralytics import YOLO

print("All imports successful ✅")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
All imports successful ✅


In [6]:
#authenticate GCS
!gcloud auth login --no-launch-browser

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=QGxtUrsn13tHUuuivlcQJrMUyaI0Qe&prompt=consent&token_usage=remote&access_type=offline&code_challenge=Kah34y70aBQmPQQWJFiJkzAMl42i2GzKWTAwftK_Cic&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AfrIepCZPnSPF7b183HWCYi3w2bQzP7Q8-ZMljM37qjfGICZaYXoa94-FI_PjR3v9y7N2g

You are now logged in as [suh2162674@maricopa.edu].
Your current projec

In [7]:
## Check GCS access
!gcloud config set project solar-cycle-487619-t6
!gsutil ls gs://mywaymo-perdataset-2026/
print("GCS connected ✅")

Project 'solar-cycle-487619-t6' lacks an 'environment' tag. Please create or add a tag with key 'environment' and a value like 'Production', 'Development', 'Test', or 'Staging'. Add an 'environment' tag using `gcloud resource-manager tags bindings create`. See https://cloud.google.com/resource-manager/docs/creating-managing-projects#designate_project_environments_with_tags for details.
Updated property [core/project].
gs://mywaymo-perdataset-2026/samples.npy
gs://mywaymo-perdataset-2026/extracted/
gs://mywaymo-perdataset-2026/models/
gs://mywaymo-perdataset-2026/prepared/
gs://mywaymo-perdataset-2026/prepared_150/
gs://mywaymo-perdataset-2026/training/
GCS connected ✅


In [8]:
# Restore prepared dataset from GCS
from pathlib import Path
LOCAL_DATA_DIR = Path("/content/waymo_yolo")
LOCAL_DATA_DIR.mkdir(exist_ok=True)

print("Restoring...")
!gsutil -m -q cp -r gs://mywaymo-perdataset-2026/prepared_150/images /content/waymo_yolo/
!gsutil -m -q cp -r gs://mywaymo-perdataset-2026/prepared_150/labels /content/waymo_yolo/
print("Done ✅")

import os
print(f"Train: {len(os.listdir('/content/waymo_yolo/images/train'))}")
print(f"Val: {len(os.listdir('/content/waymo_yolo/images/val'))}")

Restoring...
Done ✅
Train: 23033
Val: 5759


In [9]:
#YAML
from pathlib import Path

LOCAL_DATA_DIR = Path("/content/waymo_yolo")
CLASS_NAMES = ["Vehicle", "Pedestrian", "Sign", "Cyclist"]

yaml_content = f"""
path: {LOCAL_DATA_DIR}
train: images/train
val: images/val

nc: 4
names: {CLASS_NAMES}
"""

yaml_path = LOCAL_DATA_DIR / "waymo.yaml"
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"YAML created ✅")

YAML created ✅


CUDA out of memory error — even batch=8 is too large without AMP.

AMP (Automatic Mixed Precision) actually helps with memory — it uses float16 instead of float32, cutting VRAM usage in half. The real problem was the yolo26n.pt download failing.

Fix — pre-download yolo26n.pt first, then train with AMP enabled:


In [10]:
# Pre-download yolo26n.pt for AMP check
import urllib.request
urllib.request.urlretrieve(
    "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n.pt",
    "yolo26n.pt"
)
print("yolo26n.pt downloaded ✅")

yolo26n.pt downloaded ✅


GPU idle again — training died at epoch 1 batch 514/1440, no checkpoint saved.

This keeps happening at the same point — batch ~500-700 of epoch 1. This is a consistent pattern, not random crashes.

Root cause — the !gsutil shell command inside the callback is likely causing the freeze. Shell commands (!) don't work reliably inside Python callbacks.

Fix — use subprocess instead of !gsutil:

In [11]:
#During training automatically save checkpoints to GCS after each epoch using a callback:

import subprocess

def save_checkpoint_to_gcs(trainer):
    epoch = trainer.epoch
    subprocess.run([
        "gsutil", "-q", "cp",
        "/content/runs/waymo_yolo26x_150/weights/last.pt",
        f"gs://mywaymo-perdataset-2026/checkpoints/epoch_{epoch}.pt"
    ], capture_output=True)
    print(f"Checkpoint epoch {epoch} saved to GCS ✅")

model = YOLO("yolo26x.pt")
model.add_callback("on_train_epoch_end", save_checkpoint_to_gcs)

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    name="waymo_yolo26x_150",
    project="/content/runs",
    device=0,
    workers=2,
    patience=10,
    save=True,
    save_period=1,
    plots=True,
    verbose=True,
    amp=True
)

print("Training complete ✅")


Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/waymo_yolo/waymo.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=waymo_yolo26x_150, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, p

In [1]:
# check GPU is idle
!nvidia-smi

Mon Mar  9 19:57:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

the callback saved checkpoints after every epoch to gs://mywaymo-perdataset-2026/checkpoints/epoch_N.pt.
But those are intermediate checkpoints.

We still need to save the final best.pt and last.pt separately with proper names:

In [12]:
#Save model weights to GCS
!gsutil cp /content/runs/waymo_yolo26x_150/weights/best.pt gs://mywaymo-perdataset-2026/models/waymo_yolo26x_150_best.pt
!gsutil cp /content/runs/waymo_yolo26x_150/weights/last.pt gs://mywaymo-perdataset-2026/models/waymo_yolo26x_150_last.pt
!gsutil -m -q cp /content/runs/waymo_yolo26x_150/*.png gs://mywaymo-perdataset-2026/models/yolo26x_150/
print("Model saved ✅")

Copying file:///content/runs/waymo_yolo26x_150/weights/best.pt [Content-Type=application/vnd.snesdev-page-table]...
\ [1 files][112.8 MiB/112.8 MiB]                                                
Operation completed over 1 objects/112.8 MiB.                                    
Copying file:///content/runs/waymo_yolo26x_150/weights/last.pt [Content-Type=application/vnd.snesdev-page-table]...
|
Operation completed over 1 objects/112.8 MiB.                                    
Model saved ✅


In [13]:
#Check in GCS
!gsutil ls gs://mywaymo-perdataset-2026/models/

gs://mywaymo-perdataset-2026/models/BoxF1_curve.png
gs://mywaymo-perdataset-2026/models/BoxPR_curve.png
gs://mywaymo-perdataset-2026/models/BoxP_curve.png
gs://mywaymo-perdataset-2026/models/BoxR_curve.png
gs://mywaymo-perdataset-2026/models/confusion_matrix.png
gs://mywaymo-perdataset-2026/models/confusion_matrix_normalized.png
gs://mywaymo-perdataset-2026/models/results.png
gs://mywaymo-perdataset-2026/models/waymo_yolo26x_150_best.pt
gs://mywaymo-perdataset-2026/models/waymo_yolo26x_150_last.pt
gs://mywaymo-perdataset-2026/models/waymo_yolo26x_best.pt
gs://mywaymo-perdataset-2026/models/waymo_yolo26x_last.pt
gs://mywaymo-perdataset-2026/models/waymo_yolov8m_best.pt
gs://mywaymo-perdataset-2026/models/waymo_yolov8m_last.pt
gs://mywaymo-perdataset-2026/models/waymo_yolov8m_replay_best.pt
gs://mywaymo-perdataset-2026/models/waymo_yolov8m_replay_last.pt
gs://mywaymo-perdataset-2026/models/waymo_yolov8n_best.pt
gs://mywaymo-perdataset-2026/models/waymo_yolov8n_last.pt
gs://mywaymo-perdat

## Exp 3 Results — YOLO26x 150 segments

### Final Validation Metrics

| Class | Images | Instances | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 |
|-------|--------|-----------|-----------|--------|---------|--------------|
| All | 5759 | 141612 | 0.892 | 0.687 | 0.779 | 0.510 |
| Vehicle | 5712 | 108611 | 0.921 | 0.761 | 0.845 | 0.618 |
| Pedestrian | 3095 | 32197 | 0.886 | 0.679 | 0.782 | 0.483 |
| Cyclist | 655 | 804 | 0.868 | 0.622 | 0.711 | 0.429 |

Speed: 0.1ms preprocess, 8.9ms inference per image
Training time: 19.958 hours (50 epochs, L4 GPU)

### Comparison vs All Experiments

| Metric | Baseline YOLOv8n | Exp 1 YOLOv8m | Exp 2 Replay | Exp 3 YOLO26x 150seg |
|--------|-----------------|--------------|--------------|----------------------|
| mAP@0.5 | 0.461 | 0.736 | 0.687 | **0.779** |
| mAP@0.5:0.95 | 0.274 | 0.478 | 0.433 | **0.510** |
| Precision | 0.808 | 0.925 | 0.857 | 0.892 |
| Recall | 0.364 | 0.630 | 0.607 | **0.687** |
| Pedestrian Recall | 0.0 | 0.624 | 0.604 | **0.679** |
| Cyclist Recall | 0.0 | 0.551 | 0.515 | **0.622** |
| Inference Speed | 18.8ms | 4.8ms | 3.8ms | 8.9ms |

### Conclusion
YOLO26x on 150 segments is best model so far.
Larger architecture (58.8M parameters) with CNN+Attention
improves recall on all classes especially Cyclist (+7.1%).
Inference at 8.9ms well within 50ms production target.

Next: RF-DETR-L on 150 segments — transformer architecture.